# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook demonstrates how to load, explore, and process the FAIR⁲ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, with all references following Croissant `@id`s.### Dataset SourceThe dataset schema and metadata are defined at the following Croissant URL:https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not already installed!pip install mlcroissant

## 1. Data LoadingLoad dataset metadata and instantiate the Croissant Dataset object. This gives access to the record sets and further data layers.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json# The Croissant JSON-LD descriptor URLcroissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"# Load Croissant datasetdataset = mlc.Dataset(croissant_url)metadata = dataset.metadataprint("[Dataset Name]", getattr(metadata, 'name', ''))print("[Description]\n", getattr(metadata, 'description', ''))

## 2. Data OverviewLet's review the available **record sets**, **fields**, and their `@id`s for data exploration.We print the available record sets with their corresponding fields and `@id`s.

In [ ]:
# List available record sets with their @id and fields' @idsrecord_sets = list(dataset.record_sets)print(f"Number of record sets: {len(record_sets)}")for rs in record_sets:    print(f'---\nRecord set @id: {rs["@id"]}')    # List fields within record set    field_ids = [field['@id'] for field in rs.get('field', [])]    print(f"  Fields (@id): {field_ids}")

For detailed exploration, let's preview the first records and structure for each record set using their `@id`.

In [ ]:
# Display a preview of one record from each record setfor rs in record_sets:    print(f'----\nSample record from record set @id: {rs["@id"]}')    try:        recs = list(dataset.records(record_set=rs['@id']))        print(json.dumps(recs[0], indent=2))    except Exception as e:        print(f"[!] Could not load records: {str(e)}")

## 3. Data ExtractionExtract data from a record set into a DataFrame using the record set and field `@id`s obtained above.We'll load all record sets as DataFrames for further analysis. Replace `<record_set_id>` and `<field_id>` with your specific selection from the previous cells.

In [ ]:
df_dict = {}# Collect the @id for each record set for dynamic processingrecord_set_ids = [rs['@id'] for rs in record_sets]for rs_id in record_set_ids:    try:        records = list(dataset.records(record_set=rs_id))        df_dict[rs_id] = pd.DataFrame(records)        print(f'Loaded data for record set @id: {rs_id} (n={len(records)})')    except Exception as e:        print(f"Failed for record set {rs_id}: {e}")# Select one dataframe (the main protein abundance record set) for previewmain_rs_id = record_set_ids[0]  # Adjust if neededprint(f'Fields (DataFrame columns) for record set @id {main_rs_id}:')print(list(df_dict[main_rs_id].columns))df_dict[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)Now, let's perform typical data processing steps on the main record set. We'll demonstrate:- Selecting a numeric field (e.g., peptide count, coverage, molecular weight) by its field `@id`- Filtering based on a threshold- Normalization- Optional grouping if a categorical field is presentReplace `numeric_field_id` and `group_field_id` with an actual field `@id` from the columns output above.

In [ ]:
# Example: Choose a numeric field @id (e.g., 'PeptideCount' or 'MolecularWeight')# Please replace with an actual numeric @id from your dataset, e.g., 'http://mlcommons.org/croissant/peptide_count' or similar# List available columns again for easy reference:print('Available columns:', df_dict[main_rs_id].columns.tolist())# Try common names - adjust as neededpreferred_numeric_fields = [c for c in df_dict[main_rs_id].columns if any(x in c.lower() for x in ['coverage', 'mw', 'weight', 'peptide', 'count'])]if preferred_numeric_fields:    numeric_field_id = preferred_numeric_fields[0]  # Use first matching numeric fieldelse:    raise Exception('No numeric field found in record set.')print('Using numeric field @id:', numeric_field_id)df = df_dict[main_rs_id].copy()# Ensure numeric dtypedf[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')# Threshold for filtering (e.g., coverage > 10 or peptide count > 10)threshold = 10filtered_df = df[df[numeric_field_id] > threshold].copy()print(f"Filtered records with {numeric_field_id} > {threshold}:")print(filtered_df.head())# Normalize the numeric field (z-score)filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()print(f"Normalized {numeric_field_id} for filtered records:")print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())# Try to group by a categorical variable, e.g., 'Description' or 'SampleType' - adjust the field id as neededpossible_group_fields = [c for c in df.columns if any(x in c.lower() for x in ['desc', 'type', 'sample', 'modif', 'group'])]if possible_group_fields:    group_field_id = possible_group_fields[0]    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)    print(f"Data grouped by {group_field_id} (mean values):")    print(grouped_df.head())

## 5. VisualizationLet's visualize the distribution of the selected numeric field after filtering, and (optionally) the grouping results if a group field was detected.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snsplt.figure(figsize=(8,4))sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)plt.title(f'Distribution of {numeric_field_id} (filtered > {threshold})')plt.xlabel(numeric_field_id)plt.ylabel('Count')plt.show()if 'group_field_id' in locals():    plt.figure(figsize=(10,4))    grouped_means = grouped_df[numeric_field_id].sort_values(ascending=False)    grouped_means.plot(kind='bar')    plt.title(f'Mean {numeric_field_id} by {group_field_id}')    plt.xlabel(group_field_id)    plt.ylabel(f'Mean {numeric_field_id}')    plt.tight_layout()    plt.show()

## 6. Conclusion- Using `mlcroissant`, we accessed, filtered, and visualized protein mass spectrometry data from the FAIR⁲ Croissant package, referencing all elements by `@id`.- EDA reveals the distribution and grouping of a selected numeric feature, allowing insight into record set structure.- For advanced analysis, explore other fields, additional record sets, and custom processing pipelines!